# 04 — Mô hình 2: Pose + LSTM/GRU

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


In [ ]:
# === Thiết lập chung (Kaggle) ===
import sys, os
# Cách 1: clone repo (khuyến nghị để đồng bộ code):
# !git clone https://github.com/ducqwang/FallDetection.git /kaggle/working/FD
# REPO='/kaggle/working/FD'
# Cách 2: notebook nằm sẵn trong repo:
REPO = '..' if os.path.exists('../src') else '/kaggle/working/FD'
sys.path.insert(0, REPO)
from src.utils import set_seed, load_config, on_kaggle
cfg = load_config(os.path.join(REPO, 'configs/default.yaml'))
set_seed(cfg['seed'])
# TRÊN KAGGLE: sửa cfg['paths'] trỏ tới /kaggle/input/<dataset-slug>/...
# cfg['paths']['urfd_root'] = '/kaggle/input/urfd/...'
# cfg['paths']['le2i_root'] = '/kaggle/input/le2i/...'
print('Kaggle:', on_kaggle())


In [ ]:
!pip -q install ultralytics lap


In [ ]:
from src.data_splits import index_urfd
from src.pipeline import build_pose_samples
from src.datasets import PoseSequenceDataset, pad_collate
from src.models.model2_pose_lstm import PoseLSTM, train
from torch.utils.data import DataLoader
items = index_urfd(cfg['paths']['urfd_root'])
# TODO: tách train/val theo splits['within_urfd']; ở đây minh họa:
train_items, val_items = items[:50], items[50:]
tr = build_pose_samples(train_items, cfg['data']['seq_len'])
va = build_pose_samples(val_items, cfg['data']['seq_len'])
ltr = DataLoader(PoseSequenceDataset(tr), batch_size=32, shuffle=True, collate_fn=pad_collate)
lva = DataLoader(PoseSequenceDataset(va), batch_size=32, collate_fn=pad_collate)
in_dim = tr[0][0].shape[-1]
model = PoseLSTM(in_dim=in_dim, hidden=cfg['model2_pose_lstm']['hidden'])
train(model, ltr, lva, cfg['model2_pose_lstm'], 'outputs/model2_best.pt')
